# Tối ưu một pipeline pandas chậm

Một notebook phân tích seller của Olist có hàm `slow_seller_report` viết bằng `iterrows`. Trên
112 nghìn món hàng nó chạy vài giây; khi dữ liệu lên 1 triệu dòng thì mất vài phút và team bắt đầu
"chạy qua đêm". Việc của bạn: viết lại cho **cùng kết quả**, nhanh hơn hàng chục lần, và giảm bộ nhớ.

1. `fast_seller_report(orders, items, reviews)`: kết quả giống hệt `slow_seller_report` (cùng cột,
   index, giá trị), không vòng lặp Python theo dòng, không `apply(axis=1)`, không sửa input.
   Test chạy nó trên 1 triệu món hàng với giới hạn 20 giây.
2. `optimize_dtypes(df, max_unique_ratio=0.5)`: trả bản sao với cột chuỗi có tỉ lệ số giá trị khác
   nhau / số dòng **nhỏ hơn** `max_unique_ratio` thành `category`, cột số nguyên được downcast về
   kiểu nhỏ nhất (`pd.to_numeric(..., downcast="integer")`); cột float, datetime giữ nguyên.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` đọc Olist thật, đo thời gian. Test bỏ qua các cell này.
- Chấm: `uv run pytest tracks/data-science/modules/08-coding-for-ds/exercises/intermediate/03-pandas-optimization -q`

In [ ]:
import time

import numpy as np
import pandas as pd

## 1. Hàm chậm (đặc tả)

Đọc kỹ để hiểu định nghĩa từng cột, đặc biệt: đơn có nhiều review thì lấy review mới nhất; đơn có
nhiều món của cùng seller chỉ đếm một lần trong `n_orders`, `late_rate`, `review_score`.

In [ ]:
COLUMNS = ["n_orders", "revenue", "late_rate", "review_score", "bad_review_rate"]


def slow_seller_report(orders: pd.DataFrame, items: pd.DataFrame, reviews: pd.DataFrame) -> pd.DataFrame:
    """Bảng điểm seller. ĐÚNG nhưng CHẬM: đây là đặc tả, đừng sửa hàm này.

    Chỉ tính đơn giao xong (status delivered và có order_delivered_customer_date). Mỗi seller:
      n_orders         số đơn giao xong khác nhau có món của seller
      revenue          tổng price các món của seller trong các đơn đó
      late_rate        tỉ lệ đơn trễ (order_delivered_customer_date > order_estimated_delivery_date)
      review_score     điểm trung bình review MỚI NHẤT của các đơn có review (NaN nếu không đơn nào có)
      bad_review_rate  tỉ lệ review <= 2 trên các đơn có review (NaN nếu không đơn nào có)
    Index seller_id, sắp tăng dần.
    """
    delivered = {}
    for _, row in orders.iterrows():
        if row["order_status"] == "delivered" and pd.notna(row["order_delivered_customer_date"]):
            delivered[row["order_id"]] = row["order_delivered_customer_date"] > row["order_estimated_delivery_date"]
    latest = {}
    for _, row in reviews.iterrows():
        prev = latest.get(row["order_id"])
        if prev is None or row["review_answer_timestamp"] > prev[0]:
            latest[row["order_id"]] = (row["review_answer_timestamp"], row["review_score"])
    stats = {}
    for _, row in items.iterrows():
        if row["order_id"] not in delivered:
            continue
        s = stats.setdefault(row["seller_id"], {"orders": set(), "revenue": 0.0})
        s["orders"].add(row["order_id"])
        s["revenue"] += row["price"]
    rows = []
    for seller, s in stats.items():
        late = [delivered[o] for o in s["orders"]]
        scores = [latest[o][1] for o in s["orders"] if o in latest]
        rows.append({
            "seller_id": seller,
            "n_orders": len(s["orders"]),
            "revenue": s["revenue"],
            "late_rate": sum(late) / len(late),
            "review_score": sum(scores) / len(scores) if scores else np.nan,
            "bad_review_rate": sum(x <= 2 for x in scores) / len(scores) if scores else np.nan,
        })
    return pd.DataFrame(rows, columns=["seller_id", *COLUMNS]).set_index("seller_id").sort_index()

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
print(len(orders), len(items), len(reviews))

In [ ]:
# explore: đo hàm chậm (vài giây)
start = time.perf_counter()
slow = slow_seller_report(orders, items, reviews)
slow_seconds = time.perf_counter() - start
print(f"slow: {slow_seconds:.2f} s")
slow.sort_values("revenue", ascending=False).head()

## 2. Viết lại bằng thao tác theo cột

Chia nhỏ: (a) bảng đơn giao xong kèm cờ trễ, (b) review mới nhất mỗi đơn, (c) cặp (seller, đơn)
không trùng, (d) `merge` rồi `groupby().agg()`. Doanh thu cộng theo **món**, còn các tỉ lệ tính theo
**đơn**: đó là chỗ dễ sai nhất.

In [ ]:
def fast_seller_report(orders: pd.DataFrame, items: pd.DataFrame, reviews: pd.DataFrame) -> pd.DataFrame:
    """Kết quả giống slow_seller_report, không lặp theo dòng."""
    # TODO: (a) lọc đơn giao xong, thêm cột late bằng so sánh hai cột datetime
    # TODO: (b) review mới nhất: sort_values + drop_duplicates(keep="last")
    # TODO: (c) revenue: merge items với đơn giao xong rồi groupby seller sum price
    # TODO: (d) cặp (seller_id, order_id) không trùng, merge đơn + review, groupby().agg()
    raise NotImplementedError

In [ ]:
# explore: cùng kết quả? nhanh hơn bao nhiêu?
start = time.perf_counter()
fast = fast_seller_report(orders, items, reviews)
fast_seconds = time.perf_counter() - start
pd.testing.assert_frame_equal(fast, slow, check_dtype=False)
print(f"fast: {fast_seconds:.3f} s, nhanh hơn {slow_seconds / fast_seconds:.0f} lần")

## 3. Giảm bộ nhớ bằng dtype

pandas 3 lưu chuỗi bằng kiểu `str` (Arrow), đã gọn hơn `object` nhiều. Cột ít giá trị khác nhau
(`order_status`, `seller_id` lặp lại trên hàng trăm nghìn dòng) còn gọn hơn nữa với `category`: lưu
mỗi giá trị một lần và một mảng mã số nguyên nhỏ.

In [ ]:
def optimize_dtypes(df: pd.DataFrame, max_unique_ratio: float = 0.5) -> pd.DataFrame:
    """Bản sao: chuỗi ít giá trị khác nhau -> category, số nguyên -> kiểu nhỏ nhất. Không sửa df."""
    # TODO: duyệt cột; pd.api.types.is_string_dtype / is_integer_dtype; nunique() / len(df)
    raise NotImplementedError

In [ ]:
# explore: bộ nhớ trước / sau
for name, df in [("orders", orders), ("items", items)]:
    small = optimize_dtypes(df)
    before, after = df.memory_usage(deep=True).sum() / 1e6, small.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {before:.1f} MB -> {after:.1f} MB")
    print(small.dtypes.astype(str).to_dict())

## 4. Ghi chú (không chấm tự động)

Ghi lại 3–5 dòng: thay đổi nào đem lại nhiều tốc độ nhất? Còn chỗ nào có thể nhanh hơn nữa (polars,
DuckDB, đọc ít cột hơn)? Khi nào **không** đáng tối ưu?